In [14]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]                              # hdfs://namenode:9000
RAW = HDFS + cfg["paths"]["raw"]                     # zona raw: el CSV tal cual
CURATED = HDFS + cfg["paths"]["curated"]             # zona curada: Parquet limpio
LANG_INFO = HDFS + cfg["paths"]["language_info"]     # tabla auxiliar de idiomas
RESULTS = HDFS + cfg["paths"]["results"]             # zona de resultados

# Igual que en el Tema 5: actuamos como root en HDFS, ANTES de crear la sesión
os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-pipeline")
         .master(cfg["spark"]["master"])                             # en clase: local[*]
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [2]:
local_csv = cfg["input"]["local_csv"]
Path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path
fs = Path(RAW).getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
dest = Path(RAW + "/" + os.path.basename(local_csv))

if fs.exists(dest):
    print("El CSV ya está en HDFS:", dest)
elif os.path.exists(local_csv):
    t = time.time()
    fs.mkdirs(Path(RAW))
    fs.copyFromLocalFile(False, True, Path("file://" + local_csv), dest)
    print(f"Copiado a HDFS en {time.time() - t:.0f} s:", dest)
else:
    raise FileNotFoundError("No está en HDFS ni en data/. Descarga el CSV (Paso 3) y súbelo (Paso 4).")


El CSV ya está en HDFS: hdfs://namenode:9000/raw/wikirank/wikirank_20240801.csv


In [3]:
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")

raw = (spark.read.schema(schema)
         .option("header", True)
         .option("escape", '"')          # algunos títulos llevan comillas dentro
         .csv(RAW))
raw.printSchema()
raw.show(3, truncate=40)


root
 |-- Wikipedia_article_URL: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Page_ID: long (nullable = true)
 |-- Title: string (nullable = true)
 |-- WikiRank_score: double (nullable = true)
 |-- WikiRank_URL: string (nullable = true)

+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|                   Wikipedia_article_URL|Language|Page_ID|  Title|WikiRank_score|                            WikiRank_URL|
+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|https://ar.wikipedia.org/wiki/%D9%85%...|      ar|      7|    ماء|       85.4175|https://wikirank.net/ar/%D9%85%D8%A7%...|
|https://ar.wikipedia.org/wiki/%D8%B1%...|      ar|     38|رياضيات|        79.166|https://wikirank.net/ar/%D8%B1%D9%8A%...|
|https://ar.wikipedia.org/wiki/%D8%A5%...|      ar|     62|إستونيا|       90.6549|https://wikirank.net/ar/%D8%A5%D8%B

In [4]:
q = cfg["quality"]
t = time.time()

# Una sola pasada: total de filas, nulos por columna y scores fuera de rango
checks = raw.agg(
    F.count("*").alias("filas_total"),
    *[F.sum(F.col(c).isNull().cast("int")).alias("nulos_" + c) for c in raw.columns],
    F.sum((~F.col("WikiRank_score").between(q["score_min"], q["score_max"])).cast("int"))
     .alias("score_fuera_de_rango"),
).first().asDict()

# Duplicados: la misma clave (idioma + Page_ID) más de una vez (Tema 4: "check that the key is unique")
checks["claves_duplicadas"] = (raw.groupBy("Language", "Page_ID").count()
                                  .filter("count > 1").count())

print(f"Calidad calculada en {time.time() - t:.0f} s")
for k, v in checks.items():
    print(f"{k:30s} {v}")


Calidad calculada en 224 s
filas_total                    47019883
nulos_Wikipedia_article_URL    0
nulos_Language                 0
nulos_Page_ID                  0
nulos_Title                    0
nulos_WikiRank_score           0
nulos_WikiRank_URL             0
score_fuera_de_rango           0
claves_duplicadas              0


In [5]:
url = F.col("Wikipedia_article_URL")
curated = (raw
    .dropna(subset=["Language", "Page_ID", "WikiRank_score"])                  # filas sin un campo clave
    .filter(F.col("WikiRank_score").between(q["score_min"], q["score_max"]))  # scores imposibles
    .withColumn("Language", F.lower(F.trim("Language")))
    .withColumn("Title", F.coalesce(F.col("Title"), F.lit("")))               # título vacío en vez de nulo
    # nº de "%" en la URL = letras no latinas del título (feature para el modelo)
    .withColumn("url_pct_count",
                (F.length(url) - F.length(F.regexp_replace(url, "%", ""))).cast("int"))
    .drop("Wikipedia_article_URL", "WikiRank_URL"))                           # ver nota

if checks["claves_duplicadas"] > 0:                    # solo si la celda 4 encontró duplicados
    curated = curated.dropDuplicates(["Language", "Page_ID"])

In [6]:
part = cfg["partition_column"]
t = time.time()
(curated.repartition(part)         # un solo fichero por idioma (evita miles de ficheros pequeños)
    .write.mode("overwrite")        # overwrite: re-ejecutar no duplica (Tema 5)
    .partitionBy(part)
    .parquet(CURATED))
print(f"Escrito en {time.time() - t:.0f} s")

articles = spark.read.parquet(CURATED)
checks["filas_curadas"] = articles.count()
checks["filas_eliminadas"] = checks["filas_total"] - checks["filas_curadas"]
print("Filas curadas:", checks["filas_curadas"], "| eliminadas:", checks["filas_eliminadas"])


Escrito en 197 s
Filas curadas: 47019883 | eliminadas: 0


In [7]:
quality = spark.createDataFrame([checks])
quality.coalesce(1).write.mode("overwrite").json(RESULTS + "/quality_report")
quality.show(vertical=True)


-RECORD 0-------------------------------
 claves_duplicadas           | 0        
 filas_curadas               | 47019883 
 filas_eliminadas            | 0        
 filas_total                 | 47019883 
 nulos_Language              | 0        
 nulos_Page_ID               | 0        
 nulos_Title                 | 0        
 nulos_WikiRank_URL          | 0        
 nulos_WikiRank_score        | 0        
 nulos_Wikipedia_article_URL | 0        
 score_fuera_de_rango        | 0        



In [8]:
es = articles.filter("Language = 'es'")
es.explain()
es.count()


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [Page_ID#111L,Title#112,WikiRank_score#113,url_pct_count#114,Language#115] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [isnotnull(Language#115), (Language#115 = es)], PushedFilters: [], ReadSchema: struct<Page_ID:bigint,Title:string,WikiRank_score:double,url_pct_count:int>




1905093

**Tabla Auxiliar de Idiomas**

In [9]:
!pip install langcodes language_data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 2.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [language_data]0m [language_data]


In [10]:
import langcodes

# Códigos de Wikipedia que no son idiomas "normales"
CODE_ALIAS = {"simple": "en"}                               # Simple English -> datos del inglés
FIX_NAME = {"sh": "Serbo-Croatian", "simple": "Simple English"}
FIX_SCRIPT = {"sh": "Latn", "zh-min-nan": "Latn"}           # langcodes se equivoca en estos dos

FAMILY = {"ar": "Afro-Asiatic", "he": "Afro-Asiatic", "az": "Turkic", "kk": "Turkic", "tr": "Turkic",
          "uz": "Turkic", "et": "Uralic", "fi": "Uralic", "hu": "Uralic", "eu": "Isolate",
          "ka": "Kartvelian", "ja": "Japonic", "ko": "Koreanic", "id": "Austronesian",
          "ms": "Austronesian", "ta": "Dravidian", "th": "Kra-Dai", "vi": "Austroasiatic",
          "zh": "Sino-Tibetan", "zh-min-nan": "Sino-Tibetan", "eo": "Constructed", "vo": "Constructed"}
# El resto de idiomas del dataset son indoeuropeos
for c in ["be", "bg", "ca", "cs", "da", "de", "el", "en", "es", "fa", "fr", "gl", "hi", "hr", "hy", "it",
          "la", "lt", "nl", "nn", "no", "pl", "pt", "ro", "ru", "sh", "simple", "sk", "sl", "sr", "sv", "uk", "ur"]:
    FAMILY[c] = "Indo-European"

FSI = {"da": 1, "es": 1, "fr": 1, "it": 1, "nl": 1, "nn": 1, "no": 1, "pt": 1, "ro": 1, "sv": 1,
       "de": 2, "id": 2, "ms": 2,
       "az": 3, "be": 3, "bg": 3, "cs": 3, "el": 3, "et": 3, "fa": 3, "fi": 3, "he": 3, "hi": 3, "hr": 3,
       "hu": 3, "hy": 3, "ka": 3, "kk": 3, "lt": 3, "pl": 3, "ru": 3, "sh": 3, "sk": 3, "sl": 3, "sr": 3,
       "ta": 3, "th": 3, "tr": 3, "uk": 3, "ur": 3, "uz": 3, "vi": 3,
       "ar": 4, "ja": 4, "ko": 4, "zh": 4}

def lang_row(code):
    lang = langcodes.Language.get(CODE_ALIAS.get(code, code))
    return (code,
            FIX_NAME.get(code, lang.display_name("en")),
            round(lang.speaking_population() / 1e6, 3),        # hablantes, en millones
            FIX_SCRIPT.get(code, lang.maximize().script),       # Latn, Cyrl, Arab...
            FAMILY.get(code),
            FSI.get(code))                                      # None -> nulo si el FSI no lo clasifica

# Los 55 códigos que hay en los datos. collect() es seguro: son solo 55 filas
codes = sorted(r["Language"] for r in articles.select("Language").distinct().collect())

lang_info = spark.createDataFrame(
    [lang_row(c) for c in codes],
    "Language STRING, language_name STRING, speakers_millions DOUBLE, script STRING, family STRING, fsi_category INT")
lang_info.show(60, truncate=False)


+----------+-----------------+-----------------+------+-------------+------------+
|Language  |language_name    |speakers_millions|script|family       |fsi_category|
+----------+-----------------+-----------------+------+-------------+------------+
|ar        |Arabic           |378.793          |Arab  |Afro-Asiatic |4           |
|az        |Azerbaijani      |33.804           |Latn  |Turkic       |3           |
|be        |Belarusian       |10.022           |Cyrl  |Indo-European|3           |
|bg        |Bulgarian        |7.684            |Cyrl  |Indo-European|3           |
|ca        |Catalan          |8.219            |Latn  |Indo-European|NULL        |
|cs        |Czech            |13.236           |Latn  |Indo-European|3           |
|da        |Danish           |7.248            |Latn  |Indo-European|1           |
|de        |German           |141.873          |Latn  |Indo-European|2           |
|el        |Greek            |12.249           |Grek  |Indo-European|3           |
|en 

In [11]:
lang_info.groupBy("Language").count().filter("count > 1").show()

lang_info.coalesce(1).write.mode("overwrite").parquet(LANG_INFO)
lang_info = spark.read.parquet(LANG_INFO)


+--------+-----+
|Language|count|
+--------+-----+
+--------+-----+



**Pregunta 1:** Artículos y media por idioma

In [16]:
t = time.time()
q1 = (articles.groupBy("Language")
        .agg(F.count("*").alias("n_articles"),
             F.round(F.avg("WikiRank_score"), 2).alias("avg_score"))
        .orderBy(F.desc("n_articles")))
q1.write.mode("overwrite").parquet(RESULTS + "/q1_stats_by_language")
print(f"P1: {time.time() - t:.1f} s")

q1 = spark.read.parquet(RESULTS + "/q1_stats_by_language")
q1.orderBy(F.desc("n_articles")).show(60)


P1: 11.9 s
+----------+----------+---------+
|  Language|n_articles|avg_score|
+----------+----------+---------+
|        en|   6860430|    24.71|
|        de|   2930971|    18.63|
|        fr|   2626210|    16.12|
|        sv|   2591505|    20.02|
|        nl|   2163715|    12.73|
|        ru|   1991787|    18.18|
|        es|   1905093|    18.77|
|        it|   1875385|    13.63|
|        pl|   1624175|    17.03|
|        zh|   1433026|    20.31|
|        ja|   1424380|    14.57|
|        uk|   1339146|    18.82|
|        vi|   1294722|    14.24|
|        ar|   1237943|    17.11|
|        pt|   1130161|    17.71|
|        fa|   1008969|    11.43|
|        ca|    756348|    21.16|
|        id|    696238|    13.93|
|        sr|    692273|     15.9|
|        ko|    679238|    10.44|
|        no|    633769|    15.53|
|        tr|    617404|    18.14|
|        fi|    577535|    17.64|
|        cs|    550666|    16.44|
|        hu|    544427|    19.04|
|        ro|    464908|    16.61|
|  

The same was implemented using Spark SQL.

In [17]:
articles.createOrReplaceTempView("articles")
spark.sql("""
    SELECT Language, COUNT(*) AS n_articles, ROUND(AVG(WikiRank_score), 2) AS avg_score
    FROM articles
    GROUP BY Language
    ORDER BY n_articles DESC
""").show(5)


+--------+----------+---------+
|Language|n_articles|avg_score|
+--------+----------+---------+
|      en|   6860430|    24.71|
|      de|   2930971|    18.63|
|      fr|   2626210|    16.12|
|      sv|   2591505|    20.02|
|      nl|   2163715|    12.73|
+--------+----------+---------+
only showing top 5 rows


**Pregunta 2:** Top 10 de cada idioma

In [18]:
w = Window.partitionBy("Language").orderBy(F.desc("WikiRank_score"))

t = time.time()
q2 = (articles
        .withColumn("rank", F.row_number().over(w))
        .filter(F.col("rank") <= 10)
        .select("Language", "rank", "Title", "WikiRank_score", "Page_ID"))
q2.write.mode("overwrite").parquet(RESULTS + "/q2_top10_by_language")
print(f"P2: {time.time() - t:.1f} s")

q2 = spark.read.parquet(RESULTS + "/q2_top10_by_language")
q2.filter("Language = 'es'").orderBy("rank").show(truncate=False)


P2: 186.2 s
+--------+----+---------------------+--------------+-------+
|Language|rank|Title                |WikiRank_score|Page_ID|
+--------+----+---------------------+--------------+-------+
|es      |1   |Apolo                |100.0         |293    |
|es      |2   |Budismo              |100.0         |317    |
|es      |3   |Brasil               |100.0         |410    |
|es      |4   |Camino de Santiago   |100.0         |765    |
|es      |5   |Heráclito            |100.0         |1429   |
|es      |6   |Imperio romano       |100.0         |1509   |
|es      |7   |Japón                |100.0         |1575   |
|es      |8   |Johann Sebastian Bach|100.0         |1591   |
|es      |9   |Orchidaceae          |100.0         |2062   |
|es      |10  |Pablo Picasso        |100.0         |2147   |
+--------+----+---------------------+--------------+-------+



**Pregunta 3:** Análisis de covarianza y correlación del título vs calidad

In [19]:
from pyspark.sql import functions as F

# Calculamos las características del título (longitud y palabras)
articles_feat = (articles
    .withColumn("title_length", F.length(F.col("Title")))
    .withColumn("title_words", F.size(F.split(F.trim(F.col("Title")), " ")))
)

# Mostramos una vista previa de los datos con sus características calculadas
articles_feat.select("Language", "Title", "title_length", "title_words", "WikiRank_score").show(5, truncate=30)

+--------+------------------------+------------+-----------+--------------+
|Language|                   Title|title_length|title_words|WikiRank_score|
+--------+------------------------+------------+-----------+--------------+
|      en|               Anarchism|           9|          1|       96.5726|
|      en|  Protolychnis marginata|          22|          2|       18.2228|
|      en|                  Albedo|           6|          1|       84.6915|
|      en|Lecithocera megalosperma|          24|          2|       12.2794|
|      en|                       A|           1|          1|       69.5654|
+--------+------------------------+------------+-----------+--------------+
only showing top 5 rows


In [20]:
t = time.time()

# Agrupamos por idioma y calculamos estadísticos relacionales contra el WikiRank_score
q3_summary = (articles_feat
    .groupBy("Language")
    .agg(
        F.count("*").alias("n_articles"),
        F.round(F.avg("WikiRank_score"), 2).alias("avg_score"),
        F.round(F.covar_samp("title_length", "WikiRank_score"), 4).alias("covar_title_score"),
        F.round(F.corr("title_length", "WikiRank_score"), 4).alias("corr_title_score"),
        F.round(F.corr("title_words", "WikiRank_score"), 4).alias("corr_words_score")
    )
    .filter(F.col("n_articles") > 1000)  # Filtramos idiomas pequeños para evitar ruido
    .orderBy(F.desc("n_articles"))
)

# Unimos con la tabla auxiliar de idiomas para enriquecer la respuesta
q3_final = q3_summary.join(F.broadcast(lang_info), "Language", "left")

# Guardamos el resultado en la zona de resultados de HDFS
q3_final.write.mode("overwrite").parquet(RESULTS + "/q3_correlation_analysis")
print(f"P3 calculada y guardada en {time.time() - t:.1f} s")

# Leemos y mostramos el resultado final
spark.read.parquet(RESULTS + "/q3_correlation_analysis").orderBy(F.desc("n_articles")).show(60)

P3 calculada y guardada en 27.5 s
+----------+----------+---------+-----------------+----------------+----------------+-----------------+-----------------+------+-------------+------------+
|  Language|n_articles|avg_score|covar_title_score|corr_title_score|corr_words_score|    language_name|speakers_millions|script|       family|fsi_category|
+----------+----------+---------+-----------------+----------------+----------------+-----------------+-----------------+------+-------------+------------+
|        en|   6860430|    24.71|          16.0473|          0.0853|          0.1041|          English|         1728.004|  Latn|Indo-European|        NULL|
|        de|   2930971|    18.63|          15.4683|          0.0918|          0.0703|           German|          141.873|  Latn|Indo-European|           2|
|        fr|   2626210|    16.12|           7.5778|          0.0514|          0.0282|           French|          332.956|  Latn|Indo-European|           1|
|        sv|   2591505|    20.

In [21]:
spark.stop()